## CNN Raw CSI Baseline

This notebook trains a small CNN on the same normalized CSI windows used by the classical ML baseline, using block split only.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder

from utils.cache import get_all_dataframes, get_cache_path, get_results_path
from utils.csi_preprocessing import process_magnitude_data
from utils.dl_pipeline import (
    append_cnn_summary_row,
    assert_block_split_identity,
    assert_window_identity,
    print_torch_environment,
    save_label_classes,
    set_reproducible_seeds,
    train_evaluate_cnn,
)
from utils.feature_pipeline import build_frequency_feature_dataframes
from utils.ml_pipeline import load_raw_csi_data
from utils.window_arrays import build_frequency_window_arrays


### Configuration

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CALIBRATION_MODE = "rssi"

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_user"

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": 60,
    "overlap_size": 30,
    "require_all_esps": True,
}

BANDS_TO_RUN = ("2.4 GHz", "5 GHz", "Fusion")
EXPECTED_SUBCARRIERS = {"2.4 GHz": 50, "5 GHz": 56}
EXPECTED_ANCHORS = {"2.4 GHz": 9, "5 GHz": 10}
ANCHOR_GROUPS = {
    "2.4 GHz": ["esp_01", "esp_02", "esp_03", "esp_04", "esp_05", "esp_07", "esp_08", "esp_09", "esp_10"],
    "5 GHz": ["esp_11", "esp_12", "esp_13", "esp_14", "esp_15", "esp_16", "esp_17", "esp_18", "esp_19", "esp_20"],
}

BLOCK_COUNT = 10
TEST_SIZE = 0.30
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42
FORCE_RETRAIN = False

CNN_PARAMS = {
    "epochs": 30,
    "batch_size": 64,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "dropout": 0.3,
    "max_epoch_seconds": 600.0,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "validation_size": VALIDATION_SIZE,
    "n_blocks": BLOCK_COUNT,
    "anchor_groups": ANCHOR_GROUPS,
    "torch_version": torch.__version__,
}

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)

feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path(preproc_opts, feat_opts)
summary_dir = results_dir / "summary"
plots_dir = results_dir / "plots"
for directory in (summary_dir, plots_dir, results_dir / "predictions"):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")


Feature cache path: c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\.cache\dataframes\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30
Results path: c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30


### Environment

In [3]:
DEVICE = print_torch_environment()
set_reproducible_seeds(RANDOM_STATE)


torch: 2.3.0+cpu
torch threads: 8
device: cpu
Seeds: random=42, numpy=42, torch=42



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\pedro\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\pedro\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\pedro\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

### Data

In [4]:
magnitude_data, csv_diagnostics = load_raw_csi_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
)
processed_magnitude_data, magnitude_summary = process_magnitude_data(
    magnitude_data,
    **preproc_opts,
)
display(magnitude_summary.head())


Scenarios present: 1
Locations: 53 | Users: 6 | ESPs: 19
[Z-0 inventory]
  total Z-0 files found: 114
  breakdown by user / esp / trial:
    user=01 esp=01 trial=01: 1
    user=01 esp=02 trial=01: 1
    user=01 esp=03 trial=01: 1
    user=01 esp=04 trial=01: 1
    user=01 esp=05 trial=01: 1
    user=01 esp=07 trial=01: 1
    user=01 esp=08 trial=01: 1
    user=01 esp=09 trial=01: 1
    user=01 esp=10 trial=01: 1
    user=01 esp=11 trial=01: 1
    user=01 esp=12 trial=01: 1
    user=01 esp=13 trial=01: 1
    user=01 esp=14 trial=01: 1
    user=01 esp=15 trial=01: 1
    user=01 esp=16 trial=01: 1
    user=01 esp=17 trial=01: 1
    user=01 esp=18 trial=01: 1
    user=01 esp=19 trial=01: 1
    user=01 esp=20 trial=01: 1
    user=02 esp=01 trial=01: 1
    user=02 esp=02 trial=01: 1
    user=02 esp=03 trial=01: 1
    user=02 esp=04 trial=01: 1
    user=02 esp=05 trial=01: 1
    user=02 esp=07 trial=01: 1
    user=02 esp=08 trial=01: 1
    user=02 esp=09 trial=01: 1
    user=02 esp=10 trial=0

,scenario,location,user,esp,trial,samples,subcarriers,normalization,baseline_scope
0,1,A-13,01,01,01,2092,50,empty_baseline,per_user
1,1,A-13,01,02,01,2148,50,empty_baseline,per_user
2,1,A-13,01,03,01,1922,50,empty_baseline,per_user
3,1,A-13,01,04,01,2049,50,empty_baseline,per_user
4,1,A-13,01,05,01,2079,50,empty_baseline,per_user


In [5]:
def _build_feature_dataframes():
    return build_frequency_feature_dataframes(processed_magnitude_data, **feat_opts)

feature_dataframes = get_all_dataframes(preproc_opts, feat_opts, _build_feature_dataframes)
for band in BANDS_TO_RUN:
    df = feature_dataframes[band]
    print(f"{band}: {df.shape[0]} windows, {df.shape[1]} columns")
    print(f"{band} dataframe hash: {pd.util.hash_pandas_object(df, index=True).sum()}")


[cache hit] c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\.cache\dataframes\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30\2_4ghz.parquet
[cache hit] c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\.cache\dataframes\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30\5ghz.parquet
[cache hit] c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\.cache\dataframes\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30\fusion.parquet
2.4 GHz: 17647 windows, 2708 columns
2.4 GHz dataframe hash: 1360640742909988970
5 GHz: 19302 windows, 3368 columns
5 GHz dataframe hash: 17119679148981249108
Fusion: 17607 windows, 6068 columns
Fusion dataframe hash: 5193432006116439071


In [6]:
all_locations = sorted(
    set().union(*(set(df["location"].astype(str)) for df in feature_dataframes.values()))
)
if len(all_locations) != 52:
    raise RuntimeError(f"Expected 52 position classes, found {len(all_locations)}: {all_locations}")

label_encoder = LabelEncoder()
label_encoder.fit(all_locations)
save_label_classes(label_encoder.classes_, results_dir)
print(label_encoder.classes_)


[CNN] label classes saved to c:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30\predictions\cnn_label_classes.json
['A-1' 'A-13' 'A-14' 'A-2' 'A-5' 'B-1' 'B-10' 'B-11' 'B-12' 'B-13' 'B-14'
 'B-2' 'B-5' 'B-8' 'C-1' 'C-10' 'C-11' 'C-14' 'C-2' 'C-3' 'C-4' 'C-5'
 'C-6' 'C-7' 'C-8' 'C-9' 'D-1' 'D-2' 'D-3' 'D-4' 'D-5' 'D-6' 'D-7' 'D-8'
 'E-1' 'E-10' 'E-11' 'E-12' 'E-13' 'E-2' 'E-3' 'E-4' 'E-5' 'E-6' 'E-7'
 'E-8' 'F-10' 'F-11' 'F-12' 'F-13' 'F-5' 'F-8']


### Train And Evaluate

In [7]:
cnn_predictions_by_band = {}
cnn_metrics_by_band = {}
cnn_histories_by_band = {}

for band in BANDS_TO_RUN:
    print(f"\n=== {band} ===")
    arrays, meta = build_frequency_window_arrays(
        processed_magnitude_data,
        band,
        window_size=feat_opts["window_size"],
        overlap_size=feat_opts["overlap_size"],
        require_all_esps=feat_opts["require_all_esps"],
        preproc_opts=preproc_opts,
    )

    for branch_band, array in arrays.items():
        observed_subcarriers = int(array.shape[2])
        expected_subcarriers = EXPECTED_SUBCARRIERS[branch_band]
        observed_anchors = int(array.shape[1])
        expected_anchors = EXPECTED_ANCHORS[branch_band]
        print(
            f"{branch_band}: anchors={observed_anchors}, "
            f"subcarriers={observed_subcarriers}, windows={array.shape[0]}"
        )
        if observed_subcarriers != expected_subcarriers:
            raise RuntimeError(
                f"{branch_band} subcarrier count changed: "
                f"expected {expected_subcarriers}, observed {observed_subcarriers}"
            )
        if observed_anchors != expected_anchors:
            raise RuntimeError(
                f"{branch_band} anchor count changed: "
                f"expected {expected_anchors}, observed {observed_anchors}"
            )

    if band == "Fusion":
        n_24 = arrays["2.4 GHz"].shape[0]
        n_5 = arrays["5 GHz"].shape[0]
        if n_24 != n_5 or n_24 != len(feature_dataframes[band]):
            raise RuntimeError(
                f"Fusion N mismatch: 2.4={n_24}, 5={n_5}, ML={len(feature_dataframes[band])}"
            )
        print(f"Fusion N PASS: {n_24} windows")

    assert_window_identity(band=band, meta=meta, feature_df=feature_dataframes[band])
    assert_block_split_identity(
        band=band,
        meta=meta,
        feature_df=feature_dataframes[band],
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_blocks=BLOCK_COUNT,
    )

    predictions, metrics, history = train_evaluate_cnn(
        band=band,
        arrays=arrays,
        meta=meta,
        label_encoder=label_encoder,
        device=DEVICE,
        results_dir=results_dir,
        plots_dir=plots_dir,
        params=CNN_PARAMS,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_blocks=BLOCK_COUNT,
        val_size=VALIDATION_SIZE,
        force_retrain=FORCE_RETRAIN,
    )
    cnn_predictions_by_band[band] = predictions
    cnn_metrics_by_band[band] = metrics
    cnn_histories_by_band[band] = history
    append_cnn_summary_row(
        summary_path=summary_dir / "global_summary.csv",
        band=band,
        params=CNN_PARAMS,
        metrics=metrics,
    )
    print(f"{band} metrics: {metrics}")



=== 2.4 GHz ===
[window arrays] cache path: C:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30\window_arrays\2_4ghz
[window arrays cache hit] C:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project\results\preproc=norm-empty_baseline_scope-per_user\feat=allesps-on_win60-step30\window_arrays\2_4ghz
[window arrays] 2.4 GHz: shape=(17647, 9, 50, 60), dtype=float16
[window arrays] 2.4 GHz anchors: ['esp_01', 'esp_02', 'esp_03', 'esp_04', 'esp_05', 'esp_07', 'esp_08', 'esp_09', 'esp_10']
2.4 GHz: anchors=9, subcarriers=50, windows=17647
WINDOW IDENTITY PASS: 2.4 GHz (17647 windows)
BLOCK SPLIT IDENTITY PASS: 2.4 GHz (train=11058, test=4019)
[CNN] 2.4 GHz: parameters=76020
[CNN] 2.4 GHz epoch 01/30: train_loss=3.4482 train_acc=0.1194 val_loss=3.0301 val_acc=0.1815 seconds=224.6
[CNN] 2.4 GHz epoch 02/30: train_loss=2.8378 train_acc=0.2128

In [8]:
cnn_summary = pd.DataFrame(
    [{"dataset": band, "model": "CNN", "split": "block", **metrics} for band, metrics in cnn_metrics_by_band.items()]
)
display(cnn_summary)


,dataset,model,split,position_accuracy,macro_f1,room_accuracy,mean_distance_error,median_distance_error,rmse_distance_error,p90_distance_error,samples,majority_position_accuracy,majority_room_accuracy,fit_seconds,predict_seconds,wall_seconds,used_estimator,parameter_count,best_val_accuracy,mean_seconds_per_epoch
0,2.4 GHz,CNN,block,0.536203,0.536474,0.909181,1.480049,0.0,2.766747,4.472136,4019.0,0.022394,0.652899,4544.005538,63.319172,4607.324711,DualBandCNN,76020.0,0.571782,151.464677
1,5 GHz,CNN,block,0.424806,0.421744,0.922481,1.570204,1.0,2.502579,4.123106,4515.0,0.022591,0.658472,6042.151325,82.460177,6124.611502,DualBandCNN,76308.0,0.437109,201.402844
2,Fusion,CNN,block,0.642554,0.641890,0.955101,0.906206,0.0,1.852977,3.162278,4009.0,0.022449,0.654278,8878.443587,125.517589,9003.961176,DualBandCNN,138708.0,0.680498,295.944237
